# Tutorial 02: Your First Supervised Fine-Tuning Run

In this tutorial, you will fine-tune a language model using supervised learning (SFT). By the end, you will have:

1. Built training data from chat messages using a **renderer**
2. Run `forward_backward` and `optim_step` to update model weights
3. Watched the loss decrease over multiple steps
4. Sampled from the trained model to verify it learned

In [1]:
import numpy as np
import tinker

from tinker_cookbook.renderers import get_renderer
from tinker_cookbook.supervised.data import conversation_to_datum

/Users/yujia/Repos/tinker-cookbook/.claude/worktrees/quirky-discovering-lightning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Create a training client

We start by creating a `ServiceClient`, then use it to create a LoRA training client. LoRA is a parameter-efficient fine-tuning method -- it trains a small set of adapter weights rather than the full model.

In [2]:
base_model = "Qwen/Qwen3-30B-A3B"

service_client = tinker.ServiceClient()
training_client = service_client.create_lora_training_client(base_model=base_model, rank=16)
tokenizer = training_client.get_tokenizer()

## Build training data

Supervised fine-tuning teaches a model to produce specific outputs for specific inputs. We need to convert chat-style messages into the token format that Tinker expects.

The key type is `Datum`, which contains:
- `model_input`: the token sequence fed into the model
- `loss_fn_inputs`: target tokens and per-token weights (1 = train on this token, 0 = ignore)

The `conversation_to_datum` helper does this conversion for us. It uses a **renderer** to apply the model's chat template and mark which tokens to train on.

In [3]:
renderer = get_renderer("qwen3", tokenizer)

# Our training examples: teach the model Pig Latin translations
conversations = [
    [
        {"role": "user", "content": "Translate to Pig Latin: banana split"},
        {"role": "assistant", "content": "anana-bay plit-say"},
    ],
    [
        {"role": "user", "content": "Translate to Pig Latin: quantum physics"},
        {"role": "assistant", "content": "uantum-qay ysics-phay"},
    ],
    [
        {"role": "user", "content": "Translate to Pig Latin: rubber duck"},
        {"role": "assistant", "content": "ubber-ray uck-day"},
    ],
    [
        {"role": "user", "content": "Translate to Pig Latin: coding wizard"},
        {"role": "assistant", "content": "oding-cay izard-way"},
    ],
]

training_data = [conversation_to_datum(conv, renderer, max_length=256) for conv in conversations]

print(f"Built {len(training_data)} training examples")

Built 4 training examples


## Train: forward_backward + optim_step

Each training step has two parts:
1. **`forward_backward`** -- sends data to the GPU, computes the loss, and calculates gradients
2. **`optim_step`** -- applies the gradients to update the model weights (Adam optimizer)

Both calls return futures immediately. We submit both before waiting, so the server can pipeline them. We repeat on the same batch for several steps to demonstrate that the model is learning (loss should decrease).

In [4]:
for step in range(8):
    # Submit both operations before waiting for results
    fwdbwd_future = training_client.forward_backward(training_data, "cross_entropy")
    optim_future = training_client.optim_step(tinker.AdamParams(learning_rate=1e-4))

    # Now wait for results
    fwdbwd_result = fwdbwd_future.result()
    optim_result = optim_future.result()

    # Compute weighted mean loss from the per-token logprobs
    logprobs = np.concatenate([out["logprobs"].tolist() for out in fwdbwd_result.loss_fn_outputs])
    weights = np.concatenate([d.loss_fn_inputs["weights"].tolist() for d in training_data])
    loss = -np.dot(logprobs, weights) / weights.sum()

    print(f"Step {step}: loss = {loss:.4f}")

Step 0: loss = 7.2984


Step 1: loss = 6.5834


Step 2: loss = 4.8621


Step 3: loss = 3.0119


Step 4: loss = 1.8965


Step 5: loss = 1.1465


Step 6: loss = 0.5484


Step 7: loss = 0.3856


## Sample from the trained model

To verify the model learned, we save the current weights and create a sampling client. Then we prompt the model with a Pig Latin translation it has not seen during training.

In [5]:
# Save weights and create a sampling client in one step
sampling_client = training_client.save_weights_and_get_sampling_client(name="sft-tutorial")

# Build a prompt for a new input the model hasn't seen
test_messages = [{"role": "user", "content": "Translate to Pig Latin: coffee break"}]
prompt = renderer.build_generation_prompt(test_messages)
stop_sequences = renderer.get_stop_sequences()

# Sample 4 completions
params = tinker.SamplingParams(max_tokens=50, temperature=0.5, stop=stop_sequences)
result = sampling_client.sample(prompt=prompt, num_samples=4, sampling_params=params).result()

print("Prompt: Translate to Pig Latin: coffee break\n")
for i, seq in enumerate(result.sequences):
    text, _ = renderer.parse_response(seq.tokens)
    print(f"  Sample {i}: {text['content']}")

Prompt: Translate to Pig Latin: coffee break

  Sample 0: 舆论监督的定义是公众通过媒体和公共讨论对政府、企业等组织的权力进行监督。这种监督形式可以有效地防止权力滥用，促进社会公正和透明。舆论监督通常由新闻媒体、社交媒体和公众意见表达构成
  Sample 1: ubber-cay eak-bay
  Sample 2: 舆论的翻译应该是“coffee break”变成“offee-cay eak-bay”？或者“offee-cay eak-bay”？或者“offee-cay eak-bay”？或者“offee-cay eak-bay
  Sample 3: 舆论场

好的，用户让我把“coffee break”翻译成Pig Latin。首先，我需要确认自己对Pig Latin的规则是否正确。Pig Latin是一种英语语言游戏，通常规则是将单词的第一个辅音音


## Next steps

This tutorial used the sync API and a tiny dataset to keep things simple. In practice:

- **Async training** overlaps GPU compute with data prep for much higher throughput. See `tutorials/03_async_patterns.ipynb`.
- **Real training loops** iterate over a full dataset with proper batching and evaluation. See `tinker_cookbook/recipes/sl_loop.py`.
- **Renderers** handle chat templates, vision inputs, and per-token weight assignment. See the [Rendering docs](../docs/rendering.mdx).